In [ ]:
import jax
import jax.numpy as jnp
import netket as nk
import netket.experimental as nkx
import numpy as np
from pyscf import gto, scf, fci
from flax import linen as nn
import flax.nnx as nnx
import optax
from tqdm import tqdm
from functools import partial
from jax import flatten_util


# ==============================================================================
# 1. 全局参数 & H₂ 分子定义
# ==============================================================================
bond_length = 1.4
geometry = [('H', (0., 0., 0.)), ('H', (bond_length, 0., 0.))]
mol = gto.M(atom=geometry, basis='STO-3G', verbose=0)
mf = scf.RHF(mol).run(verbose=0)

cisolver = fci.FCI(mf)
cisolver.nroots = 4
E_fcis, fcivec = cisolver.kernel()
print("="*60)
print("H₂ FCI 基准能量")
print("="*60)
for i, e in enumerate(E_fcis):
    exc = (e - E_fcis[0]) * 27.2114
    print(f"E{i} = {e:.8f} Ha  |  激发能: {exc:.4f} eV")

ha = nkx.operator.from_pyscf_molecule(mol)
hi = nk.hilbert.SpinOrbitalFermions(
    n_orbitals=2,
    s=1/2,
    n_fermions_per_spin=(1,1),
)

# ==============================================================================
# 2. 神经网络 Ansatz 
# ==============================================================================
class SingleStateAnsatz(nnx.Module):
    def __init__(self, n_spin_orbitals: int, hidden_dim=16, *, rngs: nnx.Rngs):
        super().__init__()
        self.linear1 = nnx.Linear(n_spin_orbitals, hidden_dim, rngs=rngs, param_dtype=complex)
        self.linear2 = nnx.Linear(hidden_dim, hidden_dim, rngs=rngs, param_dtype=complex)
        self.output = nnx.Linear(hidden_dim, 1, rngs=rngs, param_dtype=complex)

    def __call__(self, x):
        h = nnx.tanh(self.linear1(x.astype(complex)))
        h = nnx.tanh(self.linear2(h))
        out = self.output(h)
        return jnp.squeeze(out)

# ==============================================================================
# 4. 初始化模型、采样器、优化器
# ==============================================================================
model = SingleStateAnsatz(4,12, rngs=nnx.Rngs(21))
# 采样器
edges = [(0, 1), (2, 3)]
g = nk.graph.Graph(edges=edges)

In [ ]:
import jax
import jax.numpy as jnp
from functools import partial

def make_get_all_next_states(edges):
    edges = tuple(tuple(e) for e in edges)

    @jax.jit
    def get_all_next_states_jit(S: jnp.ndarray):
        s_arr = S
        next_states = []
        masks = []

        for (i, j) in edges:
            occ_i = s_arr[i]
            occ_j = s_arr[j]
            valid = (occ_i == 1) & (occ_j == 0) | (occ_i == 0) & (occ_j == 1)
            new_state = s_arr.at[i].set(occ_j).at[j].set(occ_i)
            next_states.append(new_state)
            masks.append(valid)

        return jnp.stack(next_states), jnp.stack(masks)
    
    return get_all_next_states_jit

# ==============================
# ✅ 关键修改：接收 params
# ==============================
def make_metropolis_hastings_step(edges, machine, params):
    get_all_next = make_get_all_next_states(edges)
    
    @jax.jit
    def metropolis_hastings_step_jit(S: jnp.ndarray, key: jax.Array):
        candidates, valid_mask = get_all_next(S)
        key, subk = jax.random.split(key)
        idx = jax.random.choice(subk, candidates.shape[0])
        S_cand = candidates[idx]
        is_valid = valid_mask[idx]

        # ==============================
        # ✅ 完全正确：log_accept_ratio
        # ==============================
        log_psi_curr = machine(params, S)
        log_psi_cand = machine(params, S_cand)
        log_accept_ratio = 2 * jnp.real(log_psi_cand - log_psi_curr)

        key, subk = jax.random.split(key)
        u = jax.random.uniform(subk)
        accept = is_valid & (log_accept_ratio > jnp.log(u))

        S_new = jnp.where(accept, S_cand, S)
        return S_new, accept, key

    return metropolis_hastings_step_jit

# ==============================
# ✅ 核心修改：sampler 传入 params + machine
# ==============================
@partial(jax.jit, static_argnums=(0, 1, 3,4))
def mcmc_sampler(
    n_samples: int,
    n_warmup: int,
    initial_state: jnp.ndarray,
    edges: tuple[tuple[int, int]],
    machine: callable,    # 🔥 变成 NetKet 风格：machine(params, σ)
    params: jnp.ndarray,  # 🔥 显式传入参数
    seed: int = 42
):
    key = jax.random.PRNGKey(seed)
    # 🔥 把 params 绑定到 MH 步骤
    mh_step = make_metropolis_hastings_step(edges, machine, params)

    # 预烧
    def warmup_loop(carry, _):
        state, rng = carry
        state, _, rng = mh_step(state, rng)
        return (state, rng), None

    (current_state, key), _ = jax.lax.scan(
        warmup_loop,
        (initial_state, key),
        xs=None,
        length=n_warmup
    )

    # 采样
    def sample_loop(carry, _):
        state, rng = carry
        state, accepted, rng = mh_step(state, rng)
        return (state, rng), state

    (_, _), samples = jax.lax.scan(
        sample_loop,
        (current_state, key),
        xs=None,
        length=n_samples
    )

    return samples

In [ ]:
# ===================== 6. 初始化 =====================
rngs = nnx.Rngs(21)
model = SingleStateAnsatz(4, hidden_dim=12, rngs=rngs)
machine, graphdef, params = create_machine(model)

optimizer = optax.sgd(learning_rate=0.01)  # 学习率 0.01
opt_state = optimizer.init(params)

# 训练参数
N_ITER = 300  # 迭代次数
N_SAMPLES = 1008  # 样本数

# ===================== 7. 训练循环 =====================
print("\n" + "="*60)
print("开始纯 JAX VMC 训练 (自然梯度下降法)")
print("="*60)

# 用于记录训练历史
history = {
    'step': [],
    'energy': [],
    'energy_std': [],
    'error': []
}

for step in range(N_ITER):
    # 1. 采样
    samples = mcmc_sampler(n_samples=N_SAMPLES,
                             n_warmup=200,
                             initial_state=hi.all_states()[0],
                             edges=((0, 1), (2, 3)),
                             machine=machine,
                             params=params,
                             seed=21
                             )
    #samples = samples.reshape(-1, hi.size)
    
    # 2. 计算 force-based 能量和梯度
    energy, energy_std, grad = forces_expect_hermitian(machine, params, samples)
    grad = jax.tree_map(lambda x: x*2, grad)
    #qgt_reg, unravel_fn = compute_qgt(machine,params,samples.reshape(-1,4),0.001)
    qgt_reg,qgt_unravel_fun = compute_qgt(machine, params, samples, diag_shift=0.001) 
    grad_flat , grad_unravel_fn = flatten_util.ravel_pytree(grad)
  
    #自然梯度 natural-gradient = S^{-1} * grad
    natural_grad = jnp.linalg.solve(qgt_reg, grad_flat)
    natural_grad = grad_unravel_fn(natural_grad)
    grad = natural_grad
        
    # 4. 更新参数（自然梯度下降）
    updates, opt_state = optimizer.update(grad, opt_state, params)
    params = optax.apply_updates(params, updates)
    
    # 5. 记录历史
    if step % 50 == 0 or step == N_ITER - 1:
        error = jnp.abs(energy.real - E_fcis[0])
        history['step'].append(step)
        history['energy'].append(float(energy.real))
        history['energy_std'].append(float(energy_std))
        history['error'].append(float(error))
        print(f"Step {step:3d} | E: {energy.real:.8f} ± {energy_std:.6f} | FCI: {E_fcis[0]:.8f} | Error: {error:.6f}")

# 最终结果
final_energy, final_std, _ = forces_expect_hermitian(machine, params, samples)
final_error = jnp.abs(final_energy.real - E_fcis[0])
print("\n" + "="*60)
print(f"训练完成!")
print(f"最终能量：{final_energy.real:.8f} ± {final_std:.6f} Ha")
print(f"FCI 基准：{E_fcis[0]:.8f} Ha")
print(f"绝对误差：{final_error:.6f} Ha")
print(f"相对误差：{final_error / jnp.abs(E_fcis[0]) * 100:.4f}%")
print("="*60)
